# F1 Strategy Optimization - Analysis Notebook

This notebook demonstrates the F1 Strategy Optimization System with:
1. Data exploration
2. ML model training
3. Strategy simulation
4. Visualization

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from app.services.data_generator import generate_dataset
from app.services.data_pipeline import F1DataPipeline
from app.models.tire_degradation import TireDegradationModel
from app.models.lap_time_predictor import LapTimePredictor
from app.models.strategy_optimizer import StrategyOptimizer
from app.core.config import CIRCUITS, TIRE_COMPOUNDS

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("Setup complete!")

## 1. Generate & Explore Data

In [ ]:
# Generate dataset
df = generate_dataset(num_samples=1000, output_path='../backend/data/f1_race_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nCircuits: {df['circuit'].unique()}")
print(f"\nTire compounds: {df['tire_compound'].unique()}")
df.head()

In [ ]:
# Visualize lap time distribution by circuit
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, circuit in enumerate(df['circuit'].unique()):
    circuit_data = df[df['circuit'] == circuit]
    axes[idx].hist(circuit_data['lap_time'], bins=30, alpha=0.7, color='red')
    axes[idx].set_title(f'{circuit}')
    axes[idx].set_xlabel('Lap Time (s)')
    axes[idx].set_ylabel('Frequency')

plt.suptitle('Lap Time Distribution by Circuit', fontsize=16)
plt.tight_layout()
plt.show()

## 2. Tire Degradation Analysis

In [ ]:
# Plot degradation curves for different tires
fig, ax = plt.subplots(figsize=(12, 6))

circuit = 'Monza'
for tire in ['soft', 'medium', 'hard']:
    config = TIRE_COMPOUNDS[tire]
    laps = range(1, 41)
    degradation = [config['degradation_rate'] * (lap ** 1.5) for lap in laps]
    ax.plot(laps, degradation, label=f'{tire.capitalize()}', linewidth=2)

ax.set_xlabel('Lap Number')
ax.set_ylabel('Degradation (seconds)')
ax.set_title(f'Tire Degradation Curves - {circuit}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Prepare data for ML
pipeline = F1DataPipeline()
(X_lt, y_lt), (X_deg, y_deg) = pipeline.prepare_for_training(df)

print(f"Lap time features shape: {X_lt.shape}")
print(f"Degradation features shape: {X_deg.shape}")

# Visualize feature correlations
plt.figure(figsize=(12, 8))
corr = X_lt.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Train ML Models

In [ ]:
# Train tire degradation model
tire_model = TireDegradationModel()
tire_metrics = tire_model.train(X_deg, y_deg)

print("Tire Degradation Model Metrics:")
for metric, value in tire_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Train lap time predictor
lap_model = LapTimePredictor()
lap_metrics = lap_model.train(X_lt, y_lt)

print("\nLap Time Predictor Metrics:")
for metric, value in lap_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Feature importance
print("\nFeature Importance:")
for feature, importance in lap_model.get_feature_importance().items():
    print(f"  {feature}: {importance:.4f}")

## 4. Strategy Optimization

In [ ]:
# Initialize optimizer
optimizer = StrategyOptimizer(lap_model, tire_model)

# Test different circuits
circuits_to_test = ['Monza', 'Silverstone', 'Monaco']

results = {}
for circuit in circuits_to_test:
    laps = CIRCUITS[circuit]['laps']
    result = optimizer.optimize(circuit, laps, 'auto', 'dry', 0.0)
    results[circuit] = result
    print(f"\n{circuit} ({laps} laps):")
    print(f"  Best: {result['best_strategy']}, Time: {result['total_time']:.1f}s")
    print(f"  Pit laps: {result['pit_laps']}")
    print(f"  Tires: {result['tires_used']}")

In [ ]:
# Visualize strategy comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (circuit, result) in enumerate(results.items()):
    strategies = list(result['strategy_comparison'].keys())
    times = [result['strategy_comparison'][s]['time'] for s in strategies]
    
    colors = ['#e10600' if s == result['best_strategy'] else '#888888' for s in strategies]
    
    axes[idx].bar(strategies, times, color=colors)
    axes[idx].set_title(f'{circuit}')
    axes[idx].set_ylabel('Total Time (s)')
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Strategy Comparison by Circuit', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Lap-by-Lap Simulation

In [ ]:
from app.services.race_simulator import RaceSimulator

# Run detailed simulation
circuit = 'Monza'
laps = 53

simulator = RaceSimulator(circuit, laps, 'soft')
simulator.initialize_race('dry', 0.0)

best_result = results[circuit]
simulator.simulate_race(
    pit_laps=best_result['pit_laps'],
    tire_strategy=best_result['tires_used']
)

summary = simulator.get_race_summary()
print(f"Simulation complete!")
print(f"  Total time: {summary['total_time']:.2f}s")
print(f"  Average lap: {summary['avg_lap_time']:.2f}s")
print(f"  Best lap: {summary['best_lap']:.2f}s")
print(f"  Max degradation: {summary['max_degradation']:.2f}s")

In [ ]:
# Plot lap times
lap_states = simulator.race_state.lap_states
lap_numbers = [s.lap_number for s in lap_states]
lap_times = [s.lap_time for s in lap_states]
degradations = [s.tire_degradation for s in lap_states]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Lap times
ax1.plot(lap_numbers, lap_times, color='#e10600', linewidth=2)
for pit_lap in best_result['pit_laps']:
    ax1.axvline(x=pit_lap, color='yellow', linestyle='--', alpha=0.7, label='Pit Stop' if pit_lap == best_result['pit_laps'][0] else '')
ax1.set_ylabel('Lap Time (s)')
ax1.set_title(f'Race Simulation - {circuit}')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Degradation
ax2.plot(lap_numbers, degradations, color='#00ff64', linewidth=2)
ax2.set_xlabel('Lap Number')
ax2.set_ylabel('Tire Degradation (s)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Explanation Analysis

In [ ]:
# Print detailed explanations
print("Strategy Explanations:")
print("=" * 60)
for circuit, result in results.items():
    print(f"\n{circuit}:")
    print(f"  {result['explanation']}")
    print(f"  Weather: {result['weather']}")
    if result.get('safety_car_laps'):
        print(f"  Safety car laps: {result['safety_car_laps']}")

## Conclusion

This notebook demonstrated:
- Data generation and cleaning
- ML model training with good performance
- Strategy optimization across multiple circuits
- Detailed race simulation
- Explainable AI for strategy decisions

The system successfully recommends optimal pit stop strategies based on tire degradation, circuit characteristics, and race conditions.